# Deep Learning Recommendation Model with Leave-One-Last-Basket Split

This notebook implements a **Neural Collaborative Filtering (NCF)** model.
## Data Splitting Strategy: Leave One Last Basket
We use a time-aware splitting strategy to simulate real-world forecasting:
1. **Group by User & Time**: All items purchased by a user at the same timestamp constitute a 'Basket' or 'Session'.
2. **Sort Baskets**: Baskets are ordered chronologically for each user.
3. **Split**:
   - **Test Set**: The *most recent (last)* basket of every user.
   - **Train Set**: All *previous* baskets of every user.

This evaluates the model's ability to predict the *next* set of items a user will buy based on their entire history.

In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import numpy as np

In [2]:
DATA_PATH = Path("../data/user_item_dl.csv")
df = pd.read_csv(DATA_PATH)

df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Data shape:", df.shape)
print(df.head())

Data shape: (387583, 3)
   user_id                              item_id           timestamp
0    17850  knitted union flag hot water bottle 2010-12-01 08:26:00
1    17850   white hanging heart t-light holder 2010-12-01 08:26:00
2    17850    glass star frosted t-light holder 2010-12-01 08:26:00
3    17850       cream cupid hearts coat hanger 2010-12-01 08:26:00
4    17850                  white metal lantern 2010-12-01 08:26:00


In [3]:
# Leave One Last Basket / Session Split

# Rank unique timestamps per user (Latest = 1, Previous = 2, 3...)
df['basket_rank'] = df.groupby('user_id')['timestamp'].rank(method='dense', ascending=False)

# Test Set: The last basket (Rank 1)
test_df = df[df['basket_rank'] == 1].copy()

# Train Set: All previous baskets (Rank > 1)
train_df = df[df['basket_rank'] > 1].copy()

print(f"Train size: {len(train_df)} ({len(train_df)/len(df):.1%})")
print(f"Test size:  {len(test_df)} ({len(test_df)/len(df):.1%})")

# Check coverage
print(f"Unique Users in Train: {train_df['user_id'].nunique()}")
print(f"Unique Users in Test:  {test_df['user_id'].nunique()}")
# Xuất Train Set ra file CSV
train_df.to_csv("train.csv", index=False)

# Xuất Test Set ra file CSV
test_df.to_csv("test.csv", index=False)

Train size: 298265 (77.0%)
Test size:  89318 (23.0%)
Unique Users in Train: 2843
Unique Users in Test:  4338


In [4]:
# 3. Encoding Users and Items
# We only fit encoders on TRAIN data to avoid data leakage
user_ids = train_df['user_id'].unique()
item_ids = train_df['item_id'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {it: j for j, it in enumerate(item_ids)}

# Handle new users/items in Test (We just drop them for this NCF demo as it can't handle cold start easily)
train_df['user_idx'] = train_df['user_id'].map(user2idx)
train_df['item_idx'] = train_df['item_id'].map(item2idx)

test_df['user_idx'] = test_df['user_id'].map(user2idx)
test_df['item_idx'] = test_df['item_id'].map(item2idx)

# Drop interactions with unseen users/items in Test
original_test_len = len(test_df)
test_df = test_df.dropna(subset=['user_idx', 'item_idx'])
test_df['user_idx'] = test_df['user_idx'].astype(int)
test_df['item_idx'] = test_df['item_idx'].astype(int)

print(f"Test items dropped due to Cold Start: {original_test_len - len(test_df)}")

n_users = len(user2idx)
n_items = len(item2idx)

print(f"Num Users: {n_users}, Num Items: {n_items}")

Test items dropped due to Cold Start: 32512
Num Users: 2843, Num Items: 3802


In [5]:
# Convert user and item IDs to indices
user2idx = {u: i for i, u in enumerate(df["user_id"].unique())}
item2idx = {it: j for j, it in enumerate(df["item_id"].unique())}   

df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["item_id"].map(item2idx)

n_users, n_items = len(user2idx), len(item2idx)
print(f"n_users = {n_users}, n_items = {n_items}")

n_users = 4338, n_items = 3866


In [6]:
# 4. PyTorch Dataset
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_idx'].values, dtype=torch.long)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx]

train_ds = InteractionDataset(train_df)
test_ds = InteractionDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False)

In [7]:
# 5. NCF Model Definition
class NCF(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, user, item):
        u_emb = self.user_embedding(user)
        i_emb = self.item_embedding(item)
        x = torch.cat([u_emb, i_emb], dim=1)
        return self.fc(x).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NCF(n_users, n_items).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [8]:
# 6. Training with BPR Loss (Bayesian Personalized Ranking)
# Replaced BCELoss with BPR which directly optimizes for Ranking (Pos > Neg)

def train_epoch(model, loader, n_neg=1):
    model.train()
    total_loss = 0
    
    # tqdm loop over batches
    for users, pos_items in tqdm(loader, desc="Training (BPR)"):
        users, pos_items = users.to(device), pos_items.to(device)
        batch_size = users.size(0)
        
        # 1. Negative Sampling (Simple Random)
        # For BPR we typically use 1 Negative per Positive (Pairwise)
        neg_items = torch.randint(0, n_items, (batch_size,)).to(device)
        
        # 2. Predictions
        # Model outputs Sigmoid [0,1]. We can treat them as probabilities.
        # BPR objective: Maximize log(sigmoid(pos_score - neg_score))
        pos_scores = model(users, pos_items)
        neg_scores = model(users, neg_items)
        
        # 3. BPR Loss Calculation
        # loss = -mean( log( sigmoid( pos - neg ) ) )
        loss = -torch.mean(torch.nn.functional.logsigmoid(pos_scores - neg_scores))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(loader)

# --- RUN TRAINING ---
EPOCHS = 50
print(f"Start Training BPR (Pairwise) for {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    # BPR typically uses 1 negative per positive for standard pairwise loss
    loss = train_epoch(model, train_loader, n_neg=1)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}")


Start Training BPR (Pairwise) for 50 epochs...


Training (BPR): 100%|██████████| 292/292 [00:06<00:00, 42.58it/s]


Epoch 1: Loss = 0.6046


Training (BPR): 100%|██████████| 292/292 [00:06<00:00, 42.74it/s]


Epoch 2: Loss = 0.5404


...

Epoch 48: Loss = 0.4928


Training (BPR): 100%|██████████| 292/292 [00:06<00:00, 47.98it/s]


Epoch 49: Loss = 0.4927


Training (BPR): 100%|██████████| 292/292 [00:06<00:00, 48.65it/s]

Epoch 50: Loss = 0.4923


In [9]:
# 7. Save Model
MODEL_PATH = Path("../models/ncf_model.pt")
MODEL_PATH.parent.mkdir(exist_ok=True, parents=True)

torch.save({
    "model": model.state_dict(),
    "user2idx": user2idx,
    "item2idx": item2idx
}, MODEL_PATH)
print("Model saved!")

Model saved!


In [10]:
# Advanced Basket Evaluation: Recall, Precision, NDCG, MAP, Coverage
import math

def calculate_basket_metrics(model, test_df, train_df, k=10, n_negatives=100):
    model.eval()
    
    # Metrics storage
    precisions = []
    recalls = []
    ndcgs = []
    maps = []
    
    # Recommended Items set (for Coverage)
    all_recommended_items = set()
    
    # 1. Group Test Items by User (Ground Truth Baskets)
    test_basket_grp = test_df.groupby('user_idx')['item_idx'].apply(list)
    test_users = test_basket_grp.index.tolist()
    
    # 2. History Lookup (to filter negatives)
    full_df = pd.concat([train_df, test_df])
    user_interacted = full_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
    
    print(f"Evaluating Full Basket Metrics @ K={k} on {len(test_users)} users...")
    
    with torch.no_grad():
        for u in tqdm(test_users):
            # Ground Truth Items (The whole basket)
            gt_items = test_basket_grp[u] # List of items user bought in last session
            gt_set = set(gt_items)
            
            if len(gt_items) == 0: continue
            
            # Negative Sampling: Sample N negatives
            interacted = user_interacted.get(u, set())
            negatives = []
            while len(negatives) < n_negatives:
                neg = np.random.randint(0, n_items)
                if neg not in interacted:
                    negatives.append(neg)
            
            # Candidate Set: GT Basket + Negatives
            candidates = list(gt_set) + negatives
            
            # Predict Scores
            u_tensor = torch.tensor([u] * len(candidates)).to(device)
            i_tensor = torch.tensor(candidates).to(device)
            scores = model(u_tensor, i_tensor)
            
            # Rank Candidates
            # Top K indices relative to 'candidates' list
            _, top_indices = torch.topk(scores, min(k, len(candidates)))
            top_indices = top_indices.cpu().numpy()
            
            # Map back to Item IDs
            recommended_items = [candidates[i] for i in top_indices]
            
            # Track for Coverage
            all_recommended_items.update(recommended_items)
            
            # --- CALCULATE METRICS ---
            
            # 1. Evaluate Hits (Intersection)
            hits = 0
            score_list = [] # For MAP/NDCG
            
            for i, item_id in enumerate(recommended_items):
                if item_id in gt_set:
                    hits += 1
                    score_list.append(1)
                else:
                    score_list.append(0)
            
            # Precision @ K
            prec = hits / k
            precisions.append(prec)
            
            # Recall @ K
            rec = hits / len(gt_set)
            recalls.append(rec)
            
            # MAP @ K (Average Precision)
            ap_sum = 0
            running_hits = 0
            for i, rel in enumerate(score_list):
                if rel == 1:
                    running_hits += 1
                    ap_sum += running_hits / (i + 1)
            
            if hits > 0:
                maps.append(ap_sum / min(k, len(gt_set))) # Standard AP definition
            else:
                maps.append(0)

            # NDCG @ K
            dcg = 0
            idcg = 0
            # DCG
            for i, rel in enumerate(score_list):
                if rel == 1:
                    dcg += 1.0 / math.log2(i + 2)
            # IDCG (Ideal ranking: all hits at top)
            num_ideal_hits = min(len(gt_set), k)
            for i in range(num_ideal_hits):
                idcg += 1.0 / math.log2(i + 2)
            
            if idcg > 0:
                ndcgs.append(dcg / idcg)
            else:
                ndcgs.append(0)

    # Catalog Coverage
    coverage = len(all_recommended_items) / n_items

    return {
        f"Precision@{k}": np.mean(precisions),
        f"Recall@{k}": np.mean(recalls),
        f"NDCG@{k}": np.mean(ndcgs),
        f"MAP@{k}": np.mean(maps),
        f"Catalog Coverage": coverage
    }

# Run Advanced Evaluation
basket_metrics = calculate_basket_metrics(model, test_df, train_df, k=10, n_negatives=100)

print("\n=== FINAL BASKET EVALUATION ===")
for metric, value in basket_metrics.items():
    if "Coverage" in metric:
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value:.4f}")

Evaluating Full Basket Metrics @ K=10 on 2841 users...


100%|██████████| 2841/2841 [00:02<00:00, 980.85it/s] 



=== FINAL BASKET EVALUATION ===
Precision@10: 0.4999
Recall@10: 0.4122
NDCG@10: 0.6524
MAP@10: 0.5314
Catalog Coverage: 30.24%


## Conclusion

Training the Neural Collaborative Filtering model converts basic user–item interactions into a deployable recommender. By learning latent factors for each user and product, the model delivers personalized suggestions that outperform simple rule‑based methods. The saved `ncf_model.pt` (with index mappings) is ready to be loaded by any API or batch job for real‑time recommendations.

In [11]:

import pandas as pd
import torch
import torch.nn as nn
import numpy as np

# --- 1. CONFIG & MODEL ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMBEDDING_DIM = 64

class NCF(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, user, item):
        u_emb = self.user_embedding(user)
        i_emb = self.item_embedding(item)
        x = torch.cat([u_emb, i_emb], dim=1)
        return self.fc(x).squeeze()

# --- 2. LOAD DATA & MODEL ---
print("Loading Model and Mappings...")
checkpoint = torch.load(r'D:\Project II- 2025.1\Recommender System\recommendation-systems-rule-base\models\ncf_model.pt', map_location=DEVICE)
user2idx = checkpoint['user2idx']
item2idx = checkpoint['item2idx']
idx2item = {v: k for k, v in item2idx.items()}

n_users = len(user2idx)
n_items = len(item2idx)

model = NCF(n_users, n_items, EMBEDDING_DIM).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Loading Rules...")
try:
    rules = pd.read_csv('data/rules.csv')
    # Build Lookup: Context Item -> List of (Target Item, Confidence)
    rule_lookup = {}
    for _, row in rules.iterrows():
        # Clean naming: remove set notation if present or just use strings
        ant = str(row['antecedent']).strip().lower()
        cons = str(row['consequent']).strip().lower()
        conf = float(row['confidence'])
        
        # Simple fix for potential CSV weirdness
        if ant not in rule_lookup:
            rule_lookup[ant] = []
        rule_lookup[ant].append((cons, conf))
        
    print(f"Loaded {len(rules)} rules.")
except Exception as e:
    print(f"Error loading rules: {e}")
    rule_lookup = {}

# --- 3. CASE STUDY FUNCTION ---
def run_case_study(user_id_real, context_item_name, alpha=0.8):
    print(f"\n--- CASE STUDY: User {user_id_real}, Context: '{context_item_name}', Alpha={alpha} ---")
    
    if user_id_real not in user2idx:
        print("User not found in model training set.")
        return
        
    u_idx = user2idx[user_id_real]
    u_tensor = torch.tensor([u_idx] * n_items).to(DEVICE)
    i_tensor = torch.arange(n_items).to(DEVICE)
    
    # 1. NCF Scores
    with torch.no_grad():
        ncf_scores = model(u_tensor, i_tensor).cpu().numpy()
    
    # 2. Rule Scores
    rule_score_map = np.zeros(n_items)
    
    # Find rules matching context
    # Context item must be in our item2idx map to be useful? Not necessarily for lookup, 
    # but the RESULT (consequent) must be recommendable (in item2idx)
    
    # Try to match context string loosely
    matches = rule_lookup.get(context_item_name, [])
    if not matches:
        # Try finding in keys partial match?
        pass
    
    print(f"  Found {len(matches)} rule-based suggestions for context '{context_item_name}'")
    
    for item_name, conf in matches:
        if item_name in item2idx:
            idx = item2idx[item_name]
            rule_score_map[idx] = conf
            
    # 3. Hybrid Scoring
    hybrid_scores = (alpha * ncf_scores) + ((1 - alpha) * rule_score_map)
    
    # 4. Top-5 Analysis
    top_indices = np.argsort(hybrid_scores)[::-1][:5]
    
    print(f"{'Rank':<5} {'Item Name':<40} {'Hybrid':<8} {'NCF':<8} {'RuleConf':<8}")
    print("-" * 75)
    for rank, idx in enumerate(top_indices, 1):
        name = idx2item[idx]
        h_score = hybrid_scores[idx]
        n_score = ncf_scores[idx]
        r_score = rule_score_map[idx]
        print(f"{rank:<5} {name[:38]:<40} {h_score:.4f}   {n_score:.4f}   {r_score:.4f}")

# --- 4. EXECUTE CASES ---
# Pick a random user and item from data/user_item_dl.csv or hardcode if known
# Example: 'white hanging heart t-light holder' is very popular
run_case_study(17850, "white hanging heart t-light holder", alpha=0.7)
run_case_study(17850, "jumbo bag red retrospot", alpha=0.5)



Loading Model and Mappings...


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 